<a href="https://colab.research.google.com/github/Hyperbar/Company_website_streamlit/blob/master/rod_crewai_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Construire un outil de recherche sur le web avec CrewAI

In [1]:
pip install crewai-tools crewai

In [2]:
from crewai_tools import ScrapeWebsiteTool, FileWriterTool, TXTSearchTool
import requests

##Étape 1 : Scraping d'un site web

In [3]:
tool = ScrapeWebsiteTool(website_url='https://fr.wikipedia.org/wiki/Intelligence_artificielle')
text = tool.run()
print(text)

Using Tool: Read website content
The following text is scraped website content:
Intelligence artificielle — Wikipédia
Aller au contenu
Menu principal
Menu principal
déplacer vers la barre latérale
masquer
Navigation
Page d’accueil Portails thématiques Article au hasard Contact
Contribuer
Débuter sur Wikipédia Aide Communauté Pages spéciales Modifications récentes
Rechercher
Rechercher
Apparence
Faire un don
Créer un compte
Se connecter
Outils personnels
Faire un don Créer un compte Se connecter
Sommaire
déplacer vers la barre latérale
masquer
Début
1
Définition
2
Techniques
Afficher / masquer la sous-section Techniques
2.1
Apprentissage automatique
2.1.1
Réseaux de neurones
2.1.1.1
Apprentissage profond
2.1.1.2
Grands modèles de langages
2.2
Recherche et optimisation
2.2.1
Recherche locale
2.2.2
Recherche dans l'espace des états
2.3
Intelligence artificielle quantique
2.4
Comportements prédéfinis (imitation d'intelligence)
2.5
Logique
2.6
Méthodes probabilistes et gestion de l'incertit

## Étape 2 : Écrire le texte extrait dans un fichier

In [4]:
file_writer_tool= FileWriterTool()

result = file_writer_tool.run(
    filename='ia.txt',
    content=text,
    directory='',
    overwrite=True
)
print(result)

Using Tool: File Writer Tool
Content successfully written to ia.txt


## Étape 3 : Configurer l'outil de recherche de texte

In [5]:
import os
from google.colab import userdata

# Récupérer la clé
api_key = userdata.get('OPENAI_API_KEY')

# Vérifier qu'elle existe et commence bien par "sk-"
print(f"Clé récupérée : {api_key[:10]}..." if api_key else "Clé non trouvée")
print(f"Longueur de la clé : {len(api_key)}" if api_key else "N/A")

# Définir la clé
os.environ['OPENAI_API_KEY'] = api_key
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

Clé récupérée : sk-proj-w8...
Longueur de la clé : 164


In [8]:
import os

os.environ['OPENAI_API_KEY'] = api_key
os.environ["OPENAI_MODEL_NAME"] = "gpt-5-nano"

# Initialiser l'outil avec un fichier texte spécifique
# afin que l'agent puisse effectuer des recherches dans le contenu du fichier texte donné.
tool = TXTSearchTool(txt='ia.txt')
print(tool)

name="Search a txt's content" description='Tool Name: Search a txt\'s content\nTool Arguments: {\'search_query\': {\'description\': "Mandatory search query you want to use to search the txt\'s content", \'type\': \'str\'}}\nTool Description: A tool that can be used to semantic search a query the ia.txt txt\'s content.' env_vars=[] args_schema=<class 'crewai_tools.tools.txt_search_tool.txt_search_tool.FixedTXTSearchToolSchema'> description_updated=False cache_function=<function BaseTool.<lambda> at 0x7a479047a700> result_as_answer=False max_usage_count=None current_usage_count=0 summarize=False similarity_threshold=0.6 limit=5 adapter=CrewAIRagAdapter(collection_name='rag_tool_collection', summarize=False, similarity_threshold=0.6, limit=5, config=None) config=None


In [7]:
pip install qdrant-client

## Étape 4 : Créer un agent pour la tâche et l'exécuter